# 02 - Validación de Calidad

**Objetivo:** Validar la calidad de los datos extraídos

**Entrada:**
- Tabla: `raw.invoice_lines`

**Salida:**
- Tabla: `raw.validacion_calidad`
- Archivos: `datos/bronze/reporte_calidad_*.csv`

In [ ]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime
import sys

# Agregar el módulo de conexión al path
sys.path.insert(0, str(Path.cwd().parent))
from conexion_sqlserver import obtener_engines

print("✓ Librerías importadas")

## 1. Obtener conexión a BD

In [ ]:
engine_origen, engine_destino = obtener_engines()

print("✓ Conectado a SQL Server")

## 2. Crear tabla de validación

In [ ]:
with engine_destino.connect() as conn:
    # Crear tabla de validación
    conn.execute("""
        IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'validacion_calidad' AND SCHEMA_NAME(schema_id) = 'raw')
        BEGIN
            CREATE TABLE raw.validacion_calidad (
                id INT PRIMARY KEY IDENTITY(1,1),
                lote_id VARCHAR(36) NOT NULL,
                tabla_origen VARCHAR(100) NOT NULL,
                numero_fila INT NOT NULL,
                tipo_error VARCHAR(50) NOT NULL,
                campo_con_error VARCHAR(100),
                valor_problematico VARCHAR(MAX),
                fecha_validacion DATETIME NOT NULL DEFAULT GETDATE()
            )
        END
    """)
    
    conn.commit()

print("✓ Tabla raw.validacion_calidad creada/verificada")

## 3. Leer datos RAW

In [ ]:
# Leer datos RAW
query = "SELECT * FROM raw.invoice_lines"
df_raw = pd.read_sql_query(query, engine_destino)

print(f"✓ Leídos {len(df_raw)} registros de raw.invoice_lines")

## 4. Definir reglas de validación

In [ ]:
# Campos que no pueden ser nulos
CAMPOS_REQUERIDOS = [
    'InvoiceID',
    'InvoiceLineID',
    'Quantity',
    'UnitPrice'
]

# Campos numéricos que no pueden ser negativos
CAMPOS_NO_NEGATIVOS = [
    'Quantity',
    'UnitPrice',
    'ExtendedPrice'
]

print("✓ Reglas de validación definidas")
print(f"  - Campos requeridos: {CAMPOS_REQUERIDOS}")
print(f"  - Campos no negativos: {CAMPOS_NO_NEGATIVOS}")

## 5. Validar datos

In [ ]:
# Realizar validaciones
validaciones = []

for idx, row in df_raw.iterrows():
    payload = json.loads(row['payload'])
    lote_id = row['lote_id']
    numero_fila = row['numero_fila']
    
    # Validar campos requeridos
    for campo in CAMPOS_REQUERIDOS:
        if campo in payload and (payload[campo] is None or str(payload[campo]).strip() == ''):
            validaciones.append({
                'lote_id': lote_id,
                'tabla_origen': row['tabla_origen'],
                'numero_fila': numero_fila,
                'tipo_error': 'NULO',
                'campo_con_error': campo,
                'valor_problematico': str(payload.get(campo, ''))
            })
    
    # Validar campos no negativos
    for campo in CAMPOS_NO_NEGATIVOS:
        if campo in payload and payload[campo] is not None:
            try:
                valor = float(payload[campo])
                if valor < 0:
                    validaciones.append({
                        'lote_id': lote_id,
                        'tabla_origen': row['tabla_origen'],
                        'numero_fila': numero_fila,
                        'tipo_error': 'NEGATIVO',
                        'campo_con_error': campo,
                        'valor_problematico': str(valor)
                    })
            except (ValueError, TypeError):
                pass

print(f"✓ Validación completada")
print(f"  - Errores encontrados: {len(validaciones)}")
print(f"  - Registros OK: {len(df_raw) - len(validaciones)}")

## 6. Crear registro de validación para cada fila

In [ ]:
# Crear un registro de validación por cada fila (OK o ERROR)
registros_validacion = []

# Añadir registros con error
for val in validaciones:
    registros_validacion.append(val)

# Añadir registros OK
filas_con_error = set([v['numero_fila'] for v in validaciones])
for idx, row in df_raw.iterrows():
    if row['numero_fila'] not in filas_con_error:
        registros_validacion.append({
            'lote_id': row['lote_id'],
            'tabla_origen': row['tabla_origen'],
            'numero_fila': row['numero_fila'],
            'tipo_error': 'OK',
            'campo_con_error': None,
            'valor_problematico': None
        })

print(f"✓ Creados {len(registros_validacion)} registros de validación")

## 7. Guardar en BD

In [ ]:
# Convertir a DataFrame
df_validacion = pd.DataFrame(registros_validacion)

# Insertar en BD
df_validacion.to_sql('validacion_calidad', engine_destino, schema='raw', if_exists='append', index=False)

print(f"✓ Insertados {len(df_validacion)} registros en raw.validacion_calidad")

## 8. Guardar reporte en CSV

In [ ]:
# Guardar reporte
datos_dir = Path.cwd() / "datos" / "bronze"
datos_dir.mkdir(parents=True, exist_ok=True)

# Obtener lote_id para el nombre del archivo
lote_id = df_raw.iloc[0]['lote_id'][:8]
csv_path = datos_dir / f"reporte_calidad_LOTE_{lote_id}.csv"

df_validacion.to_csv(csv_path, index=False, encoding='utf-8')

print(f"✓ Reporte guardado: {csv_path}")
print(f"  Tamaño: {csv_path.stat().st_size / 1024:.1f} KB")

## Resumen

In [ ]:
errores_ok = len([r for r in registros_validacion if r['tipo_error'] == 'OK'])
errores_nulo = len([r for r in registros_validacion if r['tipo_error'] == 'NULO'])
errores_negativo = len([r for r in registros_validacion if r['tipo_error'] == 'NEGATIVO'])

print("\n" + "="*60)
print("RESUMEN - ETAPA 02: VALIDACION")
print("="*60)
print(f"✓ Registros validados: {len(df_raw)}")
print(f"✓ Status OK: {errores_ok}")
print(f"✓ Errores (NULO): {errores_nulo}")
print(f"✓ Errores (NEGATIVO): {errores_negativo}")
print(f"✓ Tabla BD: raw.validacion_calidad")
print(f"✓ Archivo CSV: {csv_path.name}")
print("="*60)